# APTGM - Staged Validation Experiments (T4 GPU)

**~400k params (396k) | 7,000 steps | Auto-save to Drive**

⚠️ **IMPORTANT**: Run cells in order, check acceptance criteria before proceeding!

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
results_dir = '/content/drive/MyDrive/APTGM_Results'
os.makedirs(results_dir, exist_ok=True)
print(f'Results will be saved to: {results_dir}')

In [ ]:
!git clone https://github.com/konpep-dev/APTGM.git
%cd APTGM

## 📋 PHASE 1: Validate MQAR Generator

**Acceptance Criteria:**
- ✅ Filler tokens are **random** (not all zeros)
- ✅ Query tokens appear in input_ids at correct positions
- ✅ Target tokens match values correctly

In [ ]:
!python aptgm/test_mqar.py
print('\n⚠️ CHECK ABOVE OUTPUT:')
print('1. Are filler tokens random (not all 0)?')
print('2. Do you see query tokens in the sequence?')
print('3. Are [key→value] pairs correct?')
print('\n❌ DO NOT PROCEED if any check fails!')

In [ ]:
!pip install torch matplotlib seaborn pyyaml tqdm

## ⚙️ PHASE 2: SSM-only Baseline (Context Length Sweep)

In [ ]:
import yaml
import os
import copy

# Base config: ~400k parameters, 7k training steps
base_config = {
    'data': {
        'vocab_size': 256,
        'num_kv_pairs': 8,
        'num_queries': 4
    },
    'model': {
        'd_model': 80,
        'n_layers': 4,
        'ssm_state_dim': 32,
        'n_heads': 4,
        'n_kv_heads': 4,
        'd_ff': 320,
        'dropout': 0.1
    },
    'training': {
        'batch_size': 16,
        'max_steps': 7000,
        'learning_rate': 3e-4,
        'warmup_steps': 500,
        'log_interval': 100,
        'eval_interval': 250,
        'device': 'cuda',
        'weight_decay': 0.01,
        'lambda_gate': 1.0,
        'g_star': 0.15
    }
}

os.makedirs('aptgm/configs', exist_ok=True)

# Create configs for different sequence lengths (deepcopy to avoid aliasing)
for seq_len in [64, 128, 256, 512]:
    config = copy.deepcopy(base_config)
    config['training']['seq_len'] = seq_len
    
    with open(f'aptgm/configs/ssm_seq{seq_len}.yaml', 'w') as f:
        yaml.dump(config, f)
    print(f'✅ Created config for seq_len={seq_len}')

print('\n📊 Model config: d_model=80, n_layers=4, d_ff=320')
print('   Target: ~400k params (will verify during training)')
print('   Training: 7k steps with warmup')

### SSM @ seq_len=64 (SHORT - should succeed)

In [ ]:
!python aptgm/train.py --config aptgm/configs/ssm_seq64.yaml --model_type ssm --output_dir {results_dir}/ssm_seq64

# Verify parameter count (only if training succeeded)
import torch
import json
from pathlib import Path

model_path = Path(f'{results_dir}/ssm_seq64/model.pt')
if model_path.exists():
    checkpoint = torch.load(model_path, map_location='cpu')
    # Count params from state_dict
    total_params = sum(p.numel() for p in checkpoint['model_state_dict'].values())
    print(f'\n📊 Actual SSM model parameters: {total_params:,}')
    print(f'   Expected: ~396k')
    if abs(total_params - 396000) > 50000:
        print('⚠️  WARNING: Parameter count differs significantly from expected!')
else:
    print('\n⚠️  Training failed - model.pt not found')
    print('   Check error messages above')

### SSM @ seq_len=512 (LONG - should fail/degrade)

In [ ]:
!python aptgm/train.py --config aptgm/configs/ssm_seq512.yaml --model_type ssm --output_dir {results_dir}/ssm_seq512

### ⚠️ ACCEPTANCE CHECK: SSM Context Gap

**Compare final accuracies:**
- seq_len=64: should be **high** (>70%)
- seq_len=512: should be **much lower** (<50%)

**Also verify:**
- Both models have same parameter count

This proves SSM can't handle long-range recall!

In [ ]:
import json
from pathlib import Path

# Check if results exist
hist_64_path = Path(f'{results_dir}/ssm_seq64/history.json')
hist_512_path = Path(f'{results_dir}/ssm_seq512/history.json')

if not hist_64_path.exists() or not hist_512_path.exists():
    print('❌ ERROR: History files not found!')
    print('   Make sure both SSM trainings completed successfully.')
    print(f'   seq=64:  {"✅" if hist_64_path.exists() else "❌"} {hist_64_path}')
    print(f'   seq=512: {"✅" if hist_512_path.exists() else "❌"} {hist_512_path}')
else:
    # Load histories
    with open(hist_64_path) as f:
        hist_64 = json.load(f)
    with open(hist_512_path) as f:
        hist_512 = json.load(f)

    acc_64 = hist_64['accuracy'][-1] if hist_64['accuracy'] else 0
    acc_512 = hist_512['accuracy'][-1] if hist_512['accuracy'] else 0

    print(f'SSM @ seq_len=64:  {acc_64:.1%}')
    print(f'SSM @ seq_len=512: {acc_512:.1%}')
    print(f'\nGap: {acc_64 - acc_512:.1%}')

    if acc_64 > 0.7 and acc_512 < 0.5 and (acc_64 - acc_512) > 0.3:
        print('\n✅ PHASE 2 PASSED: SSM shows clear long-context failure')
    else:
        print('\n❌ PHASE 2 FAILED: Results don\'t match expected pattern')
        print('   → Check MQAR generator or model implementation')

## 🎯 PHASE 3-4: Main Comparison @ seq_len=128

Now train all models at medium length (128) for fair comparison

In [ ]:
# Create main config with gate parameters
config = copy.deepcopy(base_config)
config['training']['seq_len'] = 128

with open('aptgm/configs/main_seq128.yaml', 'w') as f:
    yaml.dump(config, f)
print('✅ Main config created')

### Training: SSM, Attention, APTGM

In [ ]:
!python aptgm/train.py --config aptgm/configs/main_seq128.yaml --model_type ssm --output_dir {results_dir}/ssm_seq128
print('✅ SSM training complete')

In [ ]:
!python aptgm/train_attention.py --config aptgm/configs/main_seq128.yaml --output_dir {results_dir}/attention_seq128
print('✅ Attention training complete')

In [ ]:
!python aptgm/train_aptgm.py --config aptgm/configs/main_seq128.yaml --output_dir {results_dir}/aptgm_seq128
print('✅ APTGM training complete')

### ⚠️ CRITICAL CHECK: Gate Behavior

**Most important validation:**
- g_t should be **HIGH at query positions**
- g_t should be **LOW at filler positions**

This proves the gate "understands" what matters!

In [ ]:
import json
import numpy as np

with open(f'{results_dir}/aptgm_seq128/history.json') as f:
    hist = json.load(f)

if 'gate_at_queries' in hist and 'gate_at_filler' in hist:
    gate_queries = np.array(hist['gate_at_queries'])
    gate_filler = np.array(hist['gate_at_filler'])
    
    print(f'Average g_t at QUERY positions:  {gate_queries.mean():.3f}')
    print(f'Average g_t at FILLER positions: {gate_filler.mean():.3f}')
    print(f'\nDifference: {gate_queries.mean() - gate_filler.mean():.3f}')
    
    if gate_queries.mean() > gate_filler.mean() + 0.1:
        print('\n✅ GATE WORKS: Higher at queries!')
    else:
        print('\n❌ GATE FAILED: No selective behavior')
else:
    print('⚠️ Gate tracking not found in history.json')
    print('   → Check if train_aptgm.py logs gate values correctly')

## 🔄 PHASE 5: Fixed-Gate Baselines

### Falcon-H1 style (α=0.1, 0.25)

In [ ]:
!python aptgm/train_baselines.py --config aptgm/configs/main_seq128.yaml --output_dir {results_dir}/falcon_seq128
print('✅ Falcon-H1 baselines complete')

### Hard Routing (FlowHN-style)

⚠️ **TODO**: Add hard routing baseline if train_baselines.py doesn't include it

## 📊 Generate Final Plots

In [ ]:
# Summary comparison plot
!python aptgm/create_summary_plot.py --results_dir {results_dir}

# Falcon comparison plot
!python aptgm/create_falcon_plots.py --results_dir {results_dir}

print('✅ All plots generated and saved to Drive')

## 📋 Final Summary

**Check these key results:**

1. ✅ **Phase 1**: MQAR generator produces valid data
2. ✅ **Phase 2**: SSM fails on long context (64 vs 512)
3. ✅ **Phase 3**: Attention succeeds at all lengths
4. ✅ **Phase 4**: APTGM gate is HIGH at queries, LOW at filler
5. ✅ **Phase 5**: APTGM beats fixed-α baselines

All results in: `/content/drive/MyDrive/APTGM_Results/`

## ✅ Experiments Complete!

**Next steps:**
- Download plots from Google Drive
- Review gate behavior in aptgm_seq128/history.json
- Compare final accuracies across all models